In [10]:
%reload_ext autoreload
%autoreload 2
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

In [11]:
from transformers import ViTModel, ViTConfig
import torch
from torch import nn
from torch.optim import optimizer
from torch.optim import Adam

> 到https://www.kaggle.com/settings的API裡面，create new token後把下載的檔案拿來用就好

In [12]:
import configparser, os

config = configparser.ConfigParser()
config.read('config.ini')
# Load parameters from config file
#root = '/Users/leonjye/Documents/MachineLearingData'
root = config.get('DEFAULT', 'root_dir')
train_data_dir = os.path.join(root, 'CatAndDog', 'training_set')
val_data_dir = os.path.join(root, 'CatAndDog', 'test_set')

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    #使用ImageNet的均值和标准差进行归一化
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
train_set = datasets.ImageFolder(root=train_data_dir, transform=transform)
val_set = datasets.ImageFolder(root=val_data_dir, transform=transform)

# 取得每一類的 index
cat_indices = [i for i, (_, label) in enumerate(train_set.samples) if label == 0][:500]
dog_indices = [i for i, (_, label) in enumerate(train_set.samples) if label == 1][:500]
selected_indices = cat_indices + dog_indices

train_subset = Subset(train_set, selected_indices)

batch_size = 64
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)


In [14]:
next(iter(train_loader))  # Get a batch of data to check the shape

[tensor([[[[-1.6727, -1.7069, -1.7583,  ...,  1.5468,  1.5468,  1.5468],
           [-1.5014, -1.5699, -1.6384,  ...,  1.5468,  1.5468,  1.5468],
           [-1.3644, -1.3987, -1.4672,  ...,  1.5468,  1.5468,  1.5468],
           ...,
           [-1.7583, -1.7412, -1.7240,  ..., -1.4329, -1.4500, -1.4672],
           [-1.7412, -1.7412, -1.7412,  ..., -1.3987, -1.4500, -1.4672],
           [-1.7412, -1.7412, -1.7412,  ..., -1.3987, -1.4500, -1.4672]],
 
          [[-1.6506, -1.6681, -1.7206,  ...,  1.2381,  1.2381,  1.2381],
           [-1.5455, -1.5805, -1.6331,  ...,  1.2381,  1.2381,  1.2381],
           [-1.4405, -1.4755, -1.5105,  ...,  1.2381,  1.2381,  1.2381],
           ...,
           [-1.7381, -1.7206, -1.7031,  ..., -1.5105, -1.4930, -1.4755],
           [-1.7206, -1.7206, -1.7206,  ..., -1.4755, -1.4930, -1.4755],
           [-1.7206, -1.7206, -1.7206,  ..., -1.4580, -1.4930, -1.4755]],
 
          [[-1.5256, -1.5430, -1.5779,  ...,  0.9494,  0.9319,  0.9319],
           [-

In [15]:
ViTConfig()

ViTConfig {
  "attention_probs_dropout_prob": 0.0,
  "encoder_stride": 16,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 768,
  "image_size": 224,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "model_type": "vit",
  "num_attention_heads": 12,
  "num_channels": 3,
  "num_hidden_layers": 12,
  "patch_size": 16,
  "pooler_act": "tanh",
  "pooler_output_size": 768,
  "qkv_bias": true,
  "transformers_version": "4.52.4"
}

In [16]:
model_checkpoint = 'google/vit-base-patch16-224-in21k'
class ViT(nn.Module):
  def __init__(self, config=ViTConfig(), num_labels=1,
               model_checkpoint=model_checkpoint):
        super(ViT, self).__init__()
        self.vit = ViTModel.from_pretrained(model_checkpoint, add_pooling_layer=False)
        self.classifier1 = (
            nn.Linear(config.hidden_size, 128)
        )
        self.classifier2 = (
            nn.Linear(128, num_labels)
        )
        self.classifier = nn.Sequential(
            self.classifier1,
            nn.ReLU(),
            self.classifier2)
        for param in self.vit.parameters():
            param.requires_grad = False

  def forward(self, x):
    x = self.vit(x)['last_hidden_state']
    # Use the embedding of [CLS] token
    output = self.classifier(x[:, 0, :])
    output = torch.sigmoid(output)
    return output

In [17]:
import numpy as np
class Report:
    def __init__(self, n_epochs):
        self.n_epochs = n_epochs
        self.epoch = 0
        self.trn_loss = []
        self.trn_acc = []
        self.val_acc = []

    def record(self, epoch, trn_loss=None, trn_acc=None, val_acc=None, end='\n'):
        if trn_loss is not None:
            self.trn_loss.append(trn_loss)
        if trn_acc is not None:
            self.trn_acc.append(trn_acc)
        if val_acc is not None:
            self.val_acc.append(val_acc)
        print(f'Epoch {epoch}/{self.n_epochs} - '
              f'Train Loss: {np.mean(self.trn_loss):.4f}, '
              f'Train Acc: {np.mean(self.trn_acc):.4f}, '
              f'Val Acc: {np.mean(self.val_acc):.4f}', end=end)

    def report_avgs(self, epoch):
        print(f'\nEpoch {epoch} - '
              f'Avg Train Loss: {np.mean(self.trn_loss):.4f}, '
              f'Avg Train Acc: {np.mean(self.trn_acc):.4f}, '
              f'Avg Val Acc: {np.mean(self.val_acc):.4f}')
        self.trn_loss.clear()
        self.trn_acc.clear()
        self.val_acc.clear()
@torch.no_grad()
def accuracy(x, y, model):
    model.eval()
    prediction = model(x)
    is_correct = (prediction > 0.5) == y
    return is_correct.cpu().numpy().tolist()

In [ ]:
#model = ViT().to('cuda')
model = ViT()
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr= 1e-3)

n_epochs = 3
#report = Report(n_epochs)
for epoch in range(n_epochs):
    train_epoch_losses, train_epoch_accuracies = [], []
    val_epoch_accuracies = []
    n = len(train_loader)
    for ix, batch in enumerate(iter(train_loader)):
        x, y = batch
        y = y.unsqueeze(1).float()  # Ensure y is of shape (batch_size, 1)
        model.train()
        prediction = model(x)
        batch_loss = loss_fn(prediction, y)
        batch_loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        is_correct = accuracy(x, y, model)
        #report.record(epoch+(ix+1)/n, trn_loss=batch_loss, trn_acc=np.mean(is_correct), end='\r')

    n = len(val_loader)
    for ix, batch in enumerate(iter(val_loader)):
        x, y = batch
        val_is_correct = accuracy(x, y, model)
        #report.record(epoch+(ix+1)/n, val_acc=np.mean(val_is_correct), end='\r')
    print(f'Epoch {epoch+1}/{n_epochs}, '
          f'Train Loss: {batch_loss.item():.4f}, '
          f'Train Accuracy: {np.mean(is_correct):.4f}, '
          f'Validation Accuracy: {np.mean(val_is_correct):.4f}')
    #report.report_avgs(epoch+1)

Some weights of the model checkpoint at google/vit-base-patch16-224-in21k were not used when initializing ViTModel: ['pooler.dense.bias', 'pooler.dense.weight']
- This IS expected if you are initializing ViTModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing ViTModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Batch 1/500, Prediction shape: torch.Size([2, 1]), Target shape: torch.Size([2, 1])
Batch 1/500, Prediction: tensor([[0.5269],
        [0.5302]], grad_fn=<SliceBackward0>), Target: tensor([[1.],
        [1.]])
Batch 2/500, Prediction shape: torch.Size([2, 1]), Target shape: torch.Size([2, 1])
Batch 2/500, Prediction: tensor([[0.5398],
        [0.5426]], grad_fn=<SliceBackward0>), Target: tensor([[1.],
        [1.]])
Batch 3/500, Prediction shape: torch.Size([2, 1]), Target shape: torch.Size([2, 1])
Batch 3/500, Prediction: tensor([[0.5349],
        [0.5273]], grad_fn=<SliceBackward0>), Target: tensor([[0.],
        [0.]])
Batch 4/500, Prediction shape: torch.Size([2, 1]), Target shape: torch.Size([2, 1])
Batch 4/500, Prediction: tensor([[0.5600],
        [0.5414]], grad_fn=<SliceBackward0>), Target: tensor([[1.],
        [0.]])
Batch 5/500, Prediction shape: torch.Size([2, 1]), Target shape: torch.Size([2, 1])
Batch 5/500, Prediction: tensor([[0.5746],
        [0.5230]], grad_fn=<Slice